
# K-Nearest Neighbors (KNN) From Scratch

This notebook implements the KNN algorithm block by block using the assignment data.

Main steps:
1. Store the training data in memory.
2. Input and validate `k`.
3. Define the Euclidean distance formula.
4. Use the test sample `q = (20, 3.4, 161)`.
5. Calculate distance from `q` to every training sample.
6. Choose the `k` nearest neighbors.
7. Calculate weights using `1 / (1 + distance)`.
8. Predict the final label using weighted voting.


## 1. Import libraries

In [2]:
import numpy as np
import pandas as pd

## 2. Store the training data in memory

In [3]:
data = [
    [1, 21.8, 3.4, 279, 0],
    [2, 29.9, 5.8, 240, 0],
    [3, 34.4, 8.1, 285, 0],
    [4, 30.3, 8.9, 251, 1],
    [5, 36.9, 8.6, 197, 0],
    [6, 31.1, 4.1, 290, 0],
    [7, 21.0, 3.4, 268, 0],
    [8, 22.1, 8.4, 256, 0],
    [9, 25.9, 4.6, 287, 1],
    [10, 38.2, 6.5, 230, 1],
    [11, 28.1, 9.4, 208, 1],
    [12, 37.2, 4.5, 289, 1],
    [13, 19.6, 3.8, 299, 1],
    [14, 33.1, 4.6, 262, 1],
    [15, 28.3, 4.2, 161, 0],
    [16, 21.8, 5.8, 189, 1],
    [17, 19.8, 5.7, 259, 0],
    [18, 24.9, 4.6, 264, 1],
    [19, 36.1, 9.5, 259, 0],
    [20, 32.1, 4.3, 288, 0],
]

columns = ["No.", "Feature 1", "Feature 2", "Feature 3", "Label"]
df = pd.DataFrame(data, columns=columns)

df

,No.,Feature 1,Feature 2,Feature 3,Label
0,1,21.8,3.4,279,0
1,2,29.9,5.8,240,0
2,3,34.4,8.1,285,0
3,4,30.3,8.9,251,1
4,5,36.9,8.6,197,0
5,6,31.1,4.1,290,0
6,7,21.0,3.4,268,0
7,8,22.1,8.4,256,0
8,9,25.9,4.6,287,1
9,10,38.2,6.5,230,1


## 3. Separate features and labels

In [4]:
feature_columns = ["Feature 1", "Feature 2", "Feature 3"]

X = df[feature_columns].to_numpy(dtype=float)
y = df["Label"].to_numpy(dtype=int)
n = len(df)

print(f"Number of training samples, n = {n}")
print(f"Number of features per sample = {X.shape[1]}")
print(f"Labels = {np.unique(y)}")

Number of training samples, n = 20
Number of features per sample = 3
Labels = [0 1]


## 4. Input parameter `k` with validation

In [5]:
def read_k(n, default=3):
    """Read k from the user and make sure 0 < k <= n."""
    while True:
        try:
            value = input(f"Enter k (1 to {n}). Press Enter for default {default}: ")
        except Exception:
            # This fallback helps if the notebook is executed in a non-interactive environment.
            value = ""

        if value.strip() == "":
            k = default
        else:
            try:
                k = int(value)
            except ValueError:
                print("Invalid input. k must be an integer.")
                continue

        if 0 < k <= n:
            return k

        print(f"Invalid k. Please enter a number where 0 < k <= {n}.")


k = read_k(n)
print(f"Selected k = {k}")

Selected k = 3


## 5. Assign the test sample `q`

In [6]:
q = np.array([20, 3.4, 161], dtype=float)

print("Test sample q =", q)

Test sample q = [ 20.    3.4 161. ]


## 6. Define Euclidean distance

In [7]:
def euclidean_distance(x, q):
    """Calculate Euclidean distance between one training sample x and test sample q."""
    return np.sqrt(np.sum((x - q) ** 2))


# Example: distance from the first training sample to q
example_distance = euclidean_distance(X[0], q)
print(f"Distance from sample 1 to q = {example_distance:.4f}")

Distance from sample 1 to q = 118.0137


## 7. Iterate over all training samples and calculate distances

In [8]:
distances = []

for i in range(n):
    distance = euclidean_distance(X[i], q)
    distances.append(distance)

# Convert to NumPy array for easier processing
distances = np.array(distances)

print("Distances:")
print(np.round(distances, 4))

Distances:
[118.0137  79.6541 124.9218  90.7543  40.108  129.4786 107.0047  95.1547
 126.1438  71.4272  48.0688 129.1551 138.0012 101.8531   8.3385  28.1603
  98.0272 103.1235  99.5009 127.5783]


## 8. Create the distance matrix/table

In [9]:
distance_table = df.copy()
distance_table["Distance(d(x,q))"] = distances

distance_table_display = distance_table.copy()
distance_table_display["Distance(d(x,q))"] = distance_table_display["Distance(d(x,q))"].round(4)

distance_table_display

,No.,Feature 1,Feature 2,Feature 3,Label,"Distance(d(x,q))"
0,1,21.8,3.4,279,0,118.0137
1,2,29.9,5.8,240,0,79.6541
2,3,34.4,8.1,285,0,124.9218
3,4,30.3,8.9,251,1,90.7543
4,5,36.9,8.6,197,0,40.1080
5,6,31.1,4.1,290,0,129.4786
6,7,21.0,3.4,268,0,107.0047
7,8,22.1,8.4,256,0,95.1547
8,9,25.9,4.6,287,1,126.1438
9,10,38.2,6.5,230,1,71.4272


## 9. Choose the `k` nearest neighbors

In [10]:
nearest_indices = np.argsort(distances)[:k]

nearest_neighbors = df.iloc[nearest_indices].copy()
nearest_neighbors["Distance(d(x,q))"] = distances[nearest_indices]
nearest_neighbors["Weight"] = 1 / (1 + nearest_neighbors["Distance(d(x,q))"])

nearest_neighbors = nearest_neighbors.sort_values("Distance(d(x,q))").reset_index(drop=True)

nearest_neighbors_display = nearest_neighbors.copy()
nearest_neighbors_display["Distance(d(x,q))"] = nearest_neighbors_display["Distance(d(x,q))"].round(4)
nearest_neighbors_display["Weight"] = nearest_neighbors_display["Weight"].round(6)

nearest_neighbors_display

,No.,Feature 1,Feature 2,Feature 3,Label,"Distance(d(x,q))",Weight
0,15,28.3,4.2,161,0,8.3385,0.107084
1,16,21.8,5.8,189,1,28.1603,0.034293
2,5,36.9,8.6,197,0,40.1080,0.024326


## 10. Compute the label using weighted voting

In [11]:
scores = {}

for class_label in np.unique(y):
    class_weight_sum = nearest_neighbors.loc[
        nearest_neighbors["Label"] == class_label,
        "Weight"
    ].sum()
    scores[int(class_label)] = float(class_weight_sum)

score_table = pd.DataFrame({
    "Class Label": list(scores.keys()),
    "Total Weight": list(scores.values())
})
score_table["Total Weight"] = score_table["Total Weight"].round(6)

score_table

,Class Label,Total Weight
0,0,0.131410
1,1,0.034293


## 11. Predicted the label for `q`

In [12]:
best_score = max(scores.values())
best_classes = [label for label, score in scores.items() if np.isclose(score, best_score)]

if len(best_classes) == 1:
    predicted_label = best_classes[0]
else:
    # Tie-break rule: choose the label of the closest neighbor among the tied classes.
    for _, row in nearest_neighbors.iterrows():
        if int(row["Label"]) in best_classes:
            predicted_label = int(row["Label"])
            break

print(f"Predicted label for q = {q.tolist()} with k = {k}: {predicted_label}")

Predicted label for q = [20.0, 3.4, 161.0] with k = 3: 0
